# Perturb-seq pipeline — small-scale demo

This notebook runs the **perturbseq-pipeline** end to end on one lane of a human
ESC transcription-factor Perturb-seq screen and opens the resulting report.

The demo library has **416 guides against 61 target genes**, including **30
non-targeting control guides**.

What the run produces:

| Deliverable | Location |
|---|---|
| Processed `.h5ad` | `results/demo/demo_processed.h5ad` (moved to Drive if large) |
| HTML report | `results/demo/report.html` |
| All diagnostic figures | `results/demo/figures/` |
| Per-target figures (every target, not just the ones in the report) | `results/demo/figures/perturbation/per_gene/` |
| Result tables | `results/demo/tables/` |

Measured runtime on a standard Colab CPU instance: a **285 MB download**, then
**about 6 minutes** for the pipeline itself (34,012 cells in, 27,541 after QC).

## 1. Install

In [ ]:
# On Colab, install the pipeline from GitHub.
# Locally, run `pip install -e .` from the repository root instead.
!pip install -q "git+https://github.com/weili-lab/perturbseq-pipeline.git#egg=perturbseq-pipeline[demo]"

In [ ]:
import perturbseq_pipeline as psp

print("perturbseq-pipeline", psp.__version__)

## 2. Get the demo data

`fetch_demo_data.py` downloads the shared Drive folder, finds the 10x MTX
directories inside it, and writes a ready-to-run config with the real paths
filled in.

If you already have the data (for example on a mounted Google Drive), skip the
download with `--source`:

```bash
python demo/fetch_demo_data.py \
    --source "/content/drive/MyDrive/.../small_scale/raw_counts" \
    --dest demo_data
```

In [ ]:
# Get the pipeline repository, which carries the demo scripts and configs.
import os
from pathlib import Path

if not Path("perturbseq-pipeline").exists():
    !git clone -q https://github.com/weili-lab/perturbseq-pipeline.git

os.chdir("perturbseq-pipeline")
print("working directory:", Path.cwd())

In [ ]:
!python demo/fetch_demo_data.py --dest demo_data --write-config config/demo.local.yaml

## 3. Sample metadata

Any run spanning more than one lane **requires** a sample metadata file: one row
per lane, joined on `lane_id`. Every column is merged into `adata.obs` and
travels with the output `.h5ad`.

The demo ships one lane, but the metadata file lists all four lanes of the full
experiment, so adding the other lanes later needs no extra work.

In [ ]:
import pandas as pd

pd.read_csv("demo/sample_metadata.csv")

## 4. Inspect the configuration

Everything about a run lives in one YAML file — no hidden defaults. The values
worth knowing for this demo:

- `guides.min_umi: 3` and `guides.dominance_ratio: 2.0` — a cell is assigned to
  its top guide only when that guide has >= 3 UMIs and beats the runner-up
  2-fold. Otherwise the cell is *ambiguous*; with no guide counts it is
  *unassigned*. Both are reported, never dropped.
- `perturbation.controls: [ntc, other]` — perturbation strength is measured
  against **both** non-targeting cells and cells carrying other targets.
- `perturbation.top_n_report: 12` — the 12 strongest effects go inline in the
  report; the remaining targets still get their figures written to disk.

In [ ]:
print(Path("config/demo.local.yaml").read_text())

## 5. Run the pipeline

`run_pipeline` is exactly what the command line calls, so this notebook and

```bash
perturbseq-pipeline run --config config/demo.local.yaml
```

do the same work.

In [ ]:
from perturbseq_pipeline import Config, run_pipeline

cfg = Config.from_yaml("config/demo.local.yaml")

# Keep large outputs off the local disk on Colab. Set to None to keep them here.
cfg.output.large_file_dir = None

result = run_pipeline(cfg)
print()
print(result.summary())

## 6. What came out

In [ ]:
print(f"Cells analysed      : {result.n_cells:,}")
print(f"Genes               : {result.n_genes:,}")
print(f"Targets tested      : {result.n_targets_tested}")
print(f"Effective knockdowns: {result.n_effective}")
print(f"Runtime             : {result.runtime_seconds / 60:.1f} min")
print()
print(f"Report   : {result.report}")
print(f"h5ad     : {result.h5ad}")
print(f"Figures  : {result.figures_dir}")

### Perturbation strength

`log2fc` compares each target gene's own expression in perturbed cells versus
control cells. Effective CRISPR perturbation drives it **negative**; a target is
only called effective at FDR < 0.05 *and* log2FC < 0.

In [ ]:
cols = [
    "rank", "target_gene", "n_perturbed", "n_control_ntc",
    "log2fc_ntc", "pct_knockdown_ntc", "ks_fdr_ntc", "is_hit_ntc",
    "log2fc_other", "is_hit_other",
]
table = result.perturbation_table
table[[c for c in cols if c in table.columns]].head(20)

### Diagnostic figures

Every target gene gets a figure, whether or not it made the report. Figures
excluded from the report are still on disk in
`figures/perturbation/per_gene/`.

In [ ]:
from pathlib import Path

for section in sorted({p.parent for p in result.figures_dir.rglob("*.png")}):
    n = len(list(section.glob("*.png")))
    print(f"{n:3d}  {section.relative_to(result.figures_dir)}")

In [ ]:
from IPython.display import Image, display

# The strongest knockdown, as ranked in the report.
top_gene = result.perturbation_table.iloc[0]["target_gene"]
display(Image(str(result.figures_dir / "perturbation" / "per_gene" / f"perturbation_{top_gene}.png")))

In [ ]:
display(Image(str(result.figures_dir / "perturbation" / "perturbation_waterfall.png")))

## 7. Open the report

The report is a single self-contained HTML file (figures embedded), so it can be
downloaded or shared as-is.

In [ ]:
from IPython.display import HTML, IFrame

# In Colab, download it: files.download(str(result.report))
# Inline preview:
HTML(f'<a href="{result.report}" target="_blank">Open {result.report.name}</a>')

## 8. Continue in the notebook

`result.adata` is the processed object — log-normalized values in `X` and
`layers['lognorm']`, raw counts in `layers['counts']`, and guide calls in `obs`.

In [ ]:
adata = result.adata
adata

In [ ]:
adata.obs[["lane_id", "sample", "condition", "target_gene", "perturbation_class",
           "total_guide_counts", "n_guides_detected", "leiden"]].head()

In [ ]:
adata.obs["perturbation_class"].value_counts()

## Running your own data

**From 10x count matrices** — one entry per lane, plus a metadata row per lane:

```yaml
input:
  mode: mtx
  mtx_dirs:
    S1lane1: /path/to/filtered_feature_bc_matrix_S1lane1
    S1lane2: /path/to/filtered_feature_bc_matrix_S1lane2
metadata:
  file: my_samples.csv
cluster:
  batch_key: lane_id      # enables Harmony across lanes
```

**From an existing h5ad** — guide information can arrive three ways:

```yaml
input:
  mode: h5ad
  h5ad: /path/to/my_data.h5ad
  # (a) guide features already sit in var['feature_types'] — nothing more needed
  # (b) a companion guide matrix:
  guide_h5ad: /path/to/my_guides.h5ad
  # (c) a pre-computed per-cell label, e.g. a Seurat 'genotype' column:
  guide_obs_column: genotype
  # Seurat exports often keep counts in X and log values in a layer:
  normalized_layer: logcounts
```

Generate a fully-commented starting config with:

```bash
perturbseq-pipeline init-config my_run.yaml
```